### Router based RAG application
**Goal**

- Build a router based RAG application which reads data from local files, routes and queries LLM

**Steps**
- Read data files from local
- Split, embed and store it in vector store
- Route queries
- Create chains
- Ask questions
  
**Tech stack**
- LangChain
- FAISS
- Ollama
- HuggingFaceEmbeddings

### Import libraries

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

from langchain_ollama import OllamaLLM

/home/manoj/Venvs/venv-aiml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load the data

In [4]:
tech_docs = TextLoader("data/langchain_router_rag_tech.txt").load()
finance_docs = TextLoader("data/langchain_router_rag_finance.txt").load()
print(tech_docs)
print(finance_docs)

[Document(metadata={'source': 'data/langchain_router_rag_tech.txt'}, page_content='Retrieval Augmented Generation combines vector search with language models.\nLangChain provides tools to build AI pipelines.')]
[Document(metadata={'source': 'data/langchain_router_rag_finance.txt'}, page_content='Stock markets allow companies to raise capital.\nInflation impacts purchasing power and interest rates.')]


### Split documents

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

tech_chunks = splitter.split_documents(tech_docs)
finance_chunks = splitter.split_documents(finance_docs)

### Get model and embeddings

In [6]:
llm = OllamaLLM(
        model="tinyllama",
        temperature=0.3,
        num_predict=200,
        num_ctx=2048,
)

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_28947/87140909.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|███████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 425.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Create vectorstore and retriever

In [9]:
tech_db = FAISS.from_documents(tech_chunks, embeddings)
finance_db = FAISS.from_documents(finance_chunks, embeddings)

tech_retriever = tech_db.as_retriever()
finance_retriever = finance_db.as_retriever()

### Query router

In [10]:
def route_query(question):
    if "stock" in question.lower() or "finance" in question.lower():
        return "finance"
    return "tech"

router = RunnableLambda(route_query)

In [11]:
def retrieve_docs(input):
    question = input["question"]
    route = input["route"]

    if route == "finance":
        docs = finance_retriever.invoke(question)
    else:
        docs = tech_retriever.invoke(question)

    return "\n\n".join(doc.page_content for doc in docs)

retriever_runnable = RunnableLambda(retrieve_docs)

### Prompt

In [12]:
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using context.

    Context:
    {context}

    Question:
    {question}
    """
)

### Create chains

In [13]:
rag_chain = (
    RunnableParallel({
        "question": RunnablePassthrough(), 
        "route": router
    })
    | RunnableParallel({
        "context": retriever_runnable, 
        "question": RunnableLambda(lambda x: x["question"])
    })
    | prompt | llm | StrOutputParser()
)

### Ask questions

In [14]:
result = rag_chain.invoke("What is retrieval augmented generation?")
print(result)

AI assistant: Retrieval Augmented Generation (RAG) is a type of machine learning algorithm that combines vector search and language models. RAG is designed to improve the accuracy and efficiency of natural language processing (NLP) tasks, such as text summarization, question answering, and language translation. It allows for more precise and efficient search of large corpora, which can be used to generate relevant and accurate responses to user queries.


In [15]:
result = rag_chain.invoke("What are stock markets?")
print(result)

AI: 
Sure! Stock market is a system where investors buy and sell shares of publicly traded companies. It allows companies to raise capital by selling shares to the public, which helps them grow and expand their businesses. Inflation impacts purchasing power and interest rates, which can affect the value of stocks.
